This notebook focusses on the geographic and actor specific part of the analysis

In [ ]:
import pandas as pd
from pathlib import Path
# from datetime import datetime, timedelta
# import math
import plotly.express as px
# import plotly.graph_objects as go

In [3]:
dir_processed = Path('../data/processed')
dir_processed.mkdir(exist_ok=True)
dir_raw = Path('../data/raw')
dir_raw.mkdir(exist_ok=True)

In [4]:
"function resets index if new dataframes returns jumbled indexes"

def df_index(dataframe):
    rows = len(dataframe)

    dataframe.index = range(1, rows+1)
    
    return dataframe

In [5]:
dataset = pd.read_csv(dir_processed / 'dataset.csv')
dataset.index = dataset.index + 1

In [142]:
dataset_dates = dataset[['interval', 'start date', 'end date']]

dataset_dates = dataset_dates.drop_duplicates()

df_index(dataset_dates)

dataset_dates.to_csv(dir_processed/'dataset_dates.csv', index=False)

In [37]:
def color_map_grey(category):
    colors = [
        "#C5CDD5", 
        "#C4CCD0", 
        "#CBCBCB", 
        "#B8B8B8", 
        "#C1C1C1", 
        "#A9A9A9", 
        "#B0BAC5", 
        "#BBC3CD", 
        "#ACACAC", 
        "#909EAE", 
        "#B5BEC9", 
        "#808080", 
        "#6082B6", 
        "#4F4F4F",
        ] 
    return {event: colors[i] for i, event in enumerate(category)}

In [62]:
"refactored return statement after Lecture W03D04"

def color_map_pop(selected_category, category_list):
    return {event: "#5C8DC5" if category_list[i] == selected_category else "#D3D3D3" for i, event in enumerate(category_list)}

In [10]:
def event_names(df, column):
    return list(df[column].unique())

P1: Strategic Developments

In [ ]:
df_sd = (
    dataset
    .query("event_type == 'Strategic developments'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

In [ ]:
# temp_df_sd['sub_event_type'].unique()
dataset_sd = pd.merge(df_sd, dataset_dates, on='interval')

In [ ]:
sd_events = list(dataset_sd['sub_event_type'].unique())

In [ ]:
bar_sd = px.bar(
    dataset_sd,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_grey(sd_events),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

In [39]:
bar_sd.update_layout(
    title = 'Strategic developments since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_sd.update_xaxes(dtick=1)

In [ ]:
color_map_grey(sd_events)

In [40]:
bar_sd.write_html("../docs/assets/bar_sd.html")

In [41]:
bar_transfer_territory = px.bar(
    dataset_stratdev,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_pop("Non-violent transfer of territory", sd_events),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

In [42]:
bar_transfer_territory.update_layout(
    title = 'Strategic developments since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_transfer_territory.update_xaxes(dtick=1)

In [ ]:
bar_transfer_territory.write_html('../docs/assets/bar_transfer_territory.html')

In [ ]:
dataset_sd['Group/activity change'] = (
    dataset_sd["sub_event_type"]
    .apply(lambda x: 'Group/activity change' if x == 'Change to group/activity' else "Other"))

In [ ]:
dataset_sd['Transfer of territory'] = (
    dataset_sd["sub_event_type"]
    .apply(lambda x: 'Transfer of territory' if x == 'Non-violent transfer of territory' else "Other"))

In [49]:
def events_helper_columns(df, event, designation):
    df[designation] = (
    df["sub_event_type"]
    .apply(lambda x: designation if x == event else "Other"))
    return df

In [ ]:
events_helper_columns(dataset_sd, 'Arrests', 'Arrests')
dataset_sd

In [ ]:
events_helper_columns(dataset_sd, 'Change to group/activity', 'group/activity change')
dataset_sd

,interval,sub_event_type,event_count,start date,end date,Transfer of territory,group/activity change
0,0,Agreement,3,2024-12-08,2025-01-07,Other,Other
1,0,Arrests,39,2024-12-08,2025-01-07,Other,Other
2,0,Change to group/activity,115,2024-12-08,2025-01-07,Other,group/activity change
3,0,Disrupted weapons use,8,2024-12-08,2025-01-07,Other,Other
4,0,Headquarters or base established,7,2024-12-08,2025-01-07,Other,Other
...,...,...,...,...,...,...,...
117,18,Arrests,78,2026-06-19,2026-07-19,Other,Other
118,18,Change to group/activity,62,2026-06-19,2026-07-19,Other,group/activity change
119,18,Disrupted weapons use,11,2026-06-19,2026-07-19,Other,Other
120,18,Looting/property destruction,7,2026-06-19,2026-07-19,Other,Other


In [ ]:
dataset_sd_gac = (
    dataset_sd[['interval', 'group/activity change', 'event_count']]
    .groupby(['group/activity change', 'interval'], as_index=False)
    .sum()
)
dataset_sd_gac

In [ ]:
events_helper_columns(dataset_sd, "Non-violent transfer of territory", "Transfer of territory")

In [ ]:
dataset_sd_tt = (
    dataset_sd[['interval', 'Transfer of territory', 'event_count']]
    .groupby(['Transfer of territory', 'interval'], as_index=False)
    .sum()
)

In [67]:
names_sd_tt = list(dataset_sd_tt['Transfer of territory'].unique())
names_sd_tt

['Other', 'Transfer of territory']

In [117]:
bar_transfer = px.bar(
    dataset_sd_tt,
    x='interval',
    y='event_count',
    color='Transfer of territory',
    color_discrete_map=color_map_pop("Transfer of territory", ['Other', 'Transfer of territory']),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'},
    category_orders = {'Transfer of territory': ['Transfer of territory', 'Other']}
)

In [118]:
bar_transfer.update_layout(
    title = 'Non-violent transfers of territory since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_transfer.update_xaxes(dtick=1)

In [ ]:
# dataset_stratdev_gac = (
#     dataset_stratdev
#     .groupby(['interval', 'Group/activity change'], as_index=False)
#     .sum())

In [93]:
sd_events_gac = list(dataset_sd_gac['group/activity change'].unique())
sd_events_gac

['Other', 'group/activity change']

In [115]:
bar_change_group = px.bar(
    dataset_sd_gac,
    x='interval',
    y='event_count',
    color='group/activity change',
    color_discrete_map=color_map_pop("group/activity change", sd_events_gac),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'},
    category_orders = {'group/activity change': ['group/activity change', 'Other']}
)

In [116]:
bar_change_group.update_layout(
    title = 'Change to group/activity since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10,
)
bar_change_group.update_xaxes(dtick=1)

Part 2: Explosions and Remote Violence

In [ ]:
# events_helper_columns(dataset_sd, 'Explosions/Remote violence', 'explosions/remote violence')

In [122]:
dataset_erv = (
    dataset
    .query("event_type == 'Explosions/Remote violence'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

In [ ]:
dataset_erv = pd.merge(dataset_erv, dataset_dates, on='interval')
dataset_erv

In [126]:
erv_events = list(dataset_erv['sub_event_type'].unique())

In [128]:
bar_erv = px.bar(
    dataset_erv,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_grey(erv_events),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

bar_erv.update_layout(
    title = 'Explosions/remote violence since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_erv.update_xaxes(dtick=1)

In [132]:
bar_erv.write_html("../docs/assets/bar_erv.html")

Part 3: Violence against civilians

In [133]:
dataset_vac = (
    dataset
    .query("event_type == 'Violence against civilians'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

In [ ]:
dataset_vac = pd.merge(dataset_vac, dataset_dates, on='interval')
dataset_vac

In [135]:
vac_events = list(dataset_vac['sub_event_type'].unique())

In [139]:
bar_vac = px.bar(
    dataset_vac,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_pop('Abduction/forced disappearance', vac_events),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'},
    category_orders= {'sub_event_type': ['Abduction/forced disappearance', 'Attack', 'Sexual violence']}

)

bar_vac.update_layout(
    title = 'Violence against civilians since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)

bar_vac.update_xaxes(dtick=1)

In [140]:
bar_vac.write_html("../docs/assets/bar_vac.html")